# Clean & standardize the macro (F11) dataset — including the out-of-sample window

**Purpose.** Recreate *our* macro (F11) cleaning + standardization in one documented place and run it on
the released **out-of-sample** macro file, so we get standardized macro features for **Jul–Dec 2022** that
are **consistent with how the meta-model was trained**.

**What this notebook is (and isn't).**
- It reproduces the **F11 macro context** family exactly as the model was trained on it (the 45 `f11_*`
  columns in `data/features/f11_macro_context.csv`, produced by `src/stml/metamodel/macro_features.py`).
- It is **not** `data/alternate_data_cleaned.csv` (a teammate's manually-uploaded de-paired raw file with
  no z-score; no code reads it).

**The out-of-sample run — two design choices, both load-bearing:**
1. **Stitch, don't swap.** `data/OOS_additional_data.xlsx` holds *only* Jul–Dec 2022 (no rows
   `≤ 2021-07-01`, no pre-July-2022 history). Used alone it would (a) collapse the frozen z-score to
   identity — no training rows to fit `μ`/`σ` — and (b) NaN the momentum (`chg5/20/21/63` need ~200 days
   of prior history). So we **concatenate** it onto the historical workbook into one continuous series.
2. **Freeze stays at `2021-07-01`.** The model was trained on macro features standardized with the
   `≤ 2021-07-01`-frozen `μ`/`σ`. Train/serving consistency requires replaying that **exact** transform on
   the OOS rows — *not* re-freezing at 2022-06-30, *not* an expanding window. Either alternative would feed
   the model differently-scaled inputs than it learned (and re-freezing later would also leak the H1-2022
   holdout into its own scaling).

> **Scope note.** There are no OOS *signals*/OHLCV yet (both end 2022-06-30). Macro is *global* (one set of
> 45 values per date, later broadcast across instruments and filtered to nonzero-signal rows). So we emit
> the **date-indexed standardized macro panel** for H2-2022 now; the `(date, instrument)` model rows are
> formed downstream when OOS signals arrive. F11 is also 1 of 17 families — this CSV is the **macro block**
> of the OOS feature set, staged for the rest.

## §0 — Configuration & imports

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd

from stml.io import load_clean_data
from stml.metamodel import macro_features as mf


def _find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "data").is_dir():
            return p
    return start


ROOT = _find_root(Path.cwd())
os.chdir(ROOT)

# --- Inputs: historical workbook FIRST, then the released OOS workbook(s). ---
# Stitched in order; .xlsx or .csv (same paired-column layout) both accepted.
MACRO_PATHS = ["data/additional_data.xlsx", "data/OOS_additional_data.xlsx"]
FE_TRAIN_END = "2021-07-01"          # z-score frozen on rows <= this date (unchanged from training)
IN_SAMPLE_END = "2022-06-30"         # end of the released (training+val+test) window

PANEL_OUT = "results/features/f11_macro_panel.csv"          # date-indexed, full range incl. OOS
OOS_CONTEXT_OUT = "data/features/f11_macro_context_oos.csv" # OOS slice, broadcast to 11 instruments

FE_TRAIN_TS = pd.Timestamp(FE_TRAIN_END)
IN_SAMPLE_END_TS = pd.Timestamp(IN_SAMPLE_END)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
print(f"repo root    : {ROOT}")
print(f"macro sources: {MACRO_PATHS}")
print(f"FE-train end : {FE_TRAIN_END}  (frozen z-score boundary — NOT changed for OOS)")

### Curation: which series we keep, drop, and how many columns

22 named series → **read 14** (12 standalone + 2 spread-only inputs `VIX3M`/`IG_OAS`) + **8 fully dropped**.
Each standalone series and each spread emits `level` + two momentum (`chg{h}`) columns → **`12×3 + 3×3 =
45`** columns. (Some docs say "10 dropped"; that counts the 2 spread-inputs as "no standalone column" —
only **8** are fully removed.)

In [ ]:
rows = []
for name, (rcls, desc) in mf.KEEP.items():
    h1, h2 = mf.MOMENTUM[rcls]
    rows.append((name, "standalone", rcls, f"level, chg{h1}, chg{h2}", desc))
for name in mf.SPREAD_INPUTS:
    rows.append((name, "spread-input", "daily", "(used only inside spreads)", ""))
for sname, (a, b, rcls, desc) in mf.SPREADS.items():
    h1, h2 = mf.MOMENTUM[rcls]
    rows.append((f"spread_{sname} = {a}-{b}", "spread", rcls, f"level, chg{h1}, chg{h2}", desc))
for name in mf.DROPPED:
    rows.append((name, "DROPPED", "-", "-", ""))
cur = pd.DataFrame(rows, columns=["series", "role", "release_class", "emits", "captures"])
print(f"22 named -> read {len(mf.KEEP) + len(mf.SPREAD_INPUTS)} + dropped {len(mf.DROPPED)} "
      f"=> {len(mf.macro_feature_columns())} columns")
cur

## §1 — Clean & **stitch**: de-pair both files, merge per series, recover cadence

`load_macro_multi` pairs each metric's `(date@j-1, value@j)` columns in **each** file (the same logic as
`macro_features.load_macro_raw`, `.xlsx` or `.csv`), **concatenates the raw observations across files per
series** (sort, de-dup keep-last), and *then* recovers native cadence once (`daily` as-is, `weekly_eia` →
`W-FRI`, `monthly_pmi` → month-end). Because the historical (→2022-06-30) and OOS (2022-07-01→) ranges are
disjoint, this yields one continuous series per metric — exactly what the frozen z-score and the momentum
warm-up need.

In [ ]:
def _read_frame(path: str) -> pd.DataFrame:
    p = str(path)
    if p.lower().endswith(".csv"):
        return pd.read_csv(p, header=0)
    return pd.read_excel(p, engine="openpyxl", header=0)


def _obs_from_frame(frame: pd.DataFrame, name: str) -> pd.Series:
    # One series' stamp-indexed observations: value col at j, its date col at j-1.
    cols = list(frame.columns)
    j = cols.index(name)
    dates = pd.to_datetime(frame.iloc[:, j - 1], errors="coerce")
    values = pd.to_numeric(frame.iloc[:, j], errors="coerce")
    s = pd.Series(values.to_numpy(), index=pd.DatetimeIndex(dates))
    s = s[s.index.notna() & s.notna()].sort_index()
    return s[~s.index.duplicated(keep="last")]


def load_macro_multi(paths: list[str]) -> dict[str, pd.Series]:
    # Merge raw observations across files BEFORE cadence recovery, then resample once.
    frames = [_read_frame(p) for p in paths]
    out: dict[str, pd.Series] = {}
    for name, rcls in mf._all_series_classes().items():
        parts = [_obs_from_frame(fr, name) for fr in frames if name in fr.columns]
        if not parts:
            raise KeyError(f"series {name!r} not found in any of {paths}")
        obs = pd.concat(parts).sort_index()
        obs = obs[~obs.index.duplicated(keep="last")]   # keep-last if any boundary overlap
        if rcls == mf._WEEKLY_EIA:
            stamped = obs.resample("W-FRI").last().dropna()
        elif rcls == mf._MONTHLY_PMI:
            stamped = obs.resample("ME").last().dropna()
        else:
            stamped = obs
        out[name] = stamped
    return out


raw = load_macro_multi(MACRO_PATHS)
summary = pd.DataFrame(
    {
        "release_class": {k: mf._all_series_classes()[k] for k in raw},
        "n_obs": {k: len(v) for k, v in raw.items()},
        "first_stamp": {k: v.index.min().date() for k, v in raw.items()},
        "last_stamp": {k: v.index.max().date() for k, v in raw.items()},
    }
)
print(f"{len(raw)} stitched series; last stamps now extend into H2-2022:")
summary

## §2 — Point-in-time panel + raw features over the full date axis

We build the standardized panel on **every business day from the in-sample start through the last OOS
date**. Each date's feature value depends only on the macro series (not on which dates we ask for), so the
in-sample values are unchanged, and a business-day grid is a safe superset of any real OOS trade calendar
(the pipeline reindexes to actual trade dates later). Publication lags (`compute_availability`), the
200-day-buffered as-of panel (`build_applied_panel`) and the `level`+momentum assembly
(`assemble_macro_raw`) are reused verbatim.

In [ ]:
# In-sample signal calendar (for the §5 reference and instrument order); OOS span from the OOS workbook.
_ohlcv, signals = load_clean_data()
instruments = [c for c in signals.columns if c != "date"]
sig_wide = signals.set_index("date").sort_index()
in_sample_start = sig_wide.index.min()

# Nonzero-signal trade-date union — the EXACT rows the pipeline froze the z-score on (see §3).
nz_union = set()
for inst in instruments:
    s = sig_wide[inst]
    nz_union.update(s.index[s != 0])
nz_union = pd.DatetimeIndex(sorted(nz_union))

oos_dates_raw = pd.to_datetime(_read_frame(MACRO_PATHS[-1]).iloc[:, 0], errors="coerce").dropna()
OOS_MIN, OOS_MAX = oos_dates_raw.min().normalize(), oos_dates_raw.max().normalize()

all_dates = pd.bdate_range(start=in_sample_start, end=OOS_MAX)
print(f"in-sample start : {in_sample_start.date()}   OOS span: {OOS_MIN.date()} -> {OOS_MAX.date()}")
print(f"nonzero-signal in-sample dates: {len(nz_union)}")
print(f"full business-day axis: {len(all_dates)} days "
      f"({all_dates.min().date()} -> {all_dates.max().date()})")

In [ ]:
raw45 = mf.assemble_macro_raw(all_dates, raw=raw)
assert list(raw45.columns) == mf.macro_feature_columns()
oos_rows = raw45.loc[raw45.index >= OOS_MIN]
print(f"raw F11 frame: {raw45.shape}  (cols = 45 in canonical order)")
print(f"OOS rows: {len(oos_rows)}  | any NaN in OOS block? {bool(oos_rows.isna().any().any())} "
      f"(momentum warmed up via the stitched H1-2022 history)")
raw45.tail(3)

## §3 — Standardize: the **same** FE-train-frozen z-score (`≤ 2021-07-01`)

`fit_macro` freezes per-column `μ`/`σ` on the `≤ 2021-07-01` rows only — those live entirely in the
historical file, so the frozen stats are **identical to training** (e.g. VIX level `μ≈26.27, σ≈11.02`).
`transform_macro` then applies `(x − μ)/σ` to the **whole** range, H2-2022 included, so the OOS features
land on the exact scale the model learned.

In [ ]:
# Fit on the SAME rows the pipeline used: the nonzero-signal in-sample dates <= 2021-07-01
# (NOT all business days — that would shift mu/sigma and break train/serving consistency).
fit_index = nz_union[nz_union <= FE_TRAIN_TS]
raw_train = raw45.loc[fit_index]
bundle = mf.fit_macro(raw_train)
assert np.allclose(bundle.mean_, np.nanmean(raw_train.to_numpy(float), axis=0), equal_nan=True)
print(f"frozen on {len(raw_train)} nonzero-signal FE-train days "
      f"({raw_train.index.min().date()} -> {raw_train.index.max().date()})")
stats = pd.DataFrame({"frozen_mean": bundle.mean_, "frozen_std": bundle.std_}, index=bundle.feature_cols)
display(stats.head(4))

std_panel = mf.transform_macro(bundle, raw45)   # date-indexed, full range incl. OOS
print(f"standardized panel: {std_panel.shape}")
print("FE-train-slice column means (should be ~0):",
      {c: round(float(v), 6) for c, v in std_panel.loc[fit_index].mean().head(3).items()})

## §4 — Emit the standardized macro CSVs

**(1) `results/features/f11_macro_panel.csv`** — the canonical deliverable: `date` + 45 `f11_*`, full range
incl. H2-2022. This *is* "the cleaned + standardized macro features"; the meta-model pipeline broadcasts a
date-indexed macro panel like this internally.

**(2) `data/features/f11_macro_context_oos.csv`** — convenience: the OOS slice only, broadcast to all 11
instruments (training-file order), schema identical to `data/features/f11_macro_context.csv`. Rows are
**every (date × instrument)** pair (no OOS signal filter possible yet) — filter to nonzero-signal rows
once OOS signals arrive.

In [ ]:
cols = mf.macro_feature_columns()

# (1) date-indexed panel
panel = std_panel.copy()
panel.index.name = "date"
panel = panel.reset_index()[["date", *cols]]
os.makedirs(os.path.dirname(PANEL_OUT), exist_ok=True)
panel.to_csv(PANEL_OUT, index=False)
print(f"wrote {PANEL_OUT}: {panel.shape}  ({panel['date'].min().date()} -> {panel['date'].max().date()})")

# (2) OOS slice, broadcast to instruments
oos_panel = std_panel.loc[(std_panel.index >= OOS_MIN) & (std_panel.index <= OOS_MAX)]
frames = []
for inst in instruments:
    b = oos_panel.copy()
    b.index.name = "date"
    b = b.reset_index()
    b.insert(1, "instrument", inst)
    frames.append(b)
oos_ctx = pd.concat(frames, ignore_index=True)[["date", "instrument", *cols]]
os.makedirs(os.path.dirname(OOS_CONTEXT_OUT), exist_ok=True)
oos_ctx.to_csv(OOS_CONTEXT_OUT, index=False)
print(f"wrote {OOS_CONTEXT_OUT}: {oos_ctx.shape}  "
      f"({oos_panel.shape[0]} dates x {len(instruments)} instruments)")
oos_ctx.head(3)

## §5 — Consistency proof: in-sample values unchanged by the stitch + freeze

The macro values are identical across instruments on a date, so we compare the committed
`data/features/f11_macro_context.csv` (de-duplicated to one row per date) against our panel on the shared
in-sample dates. Equality there proves stitching the OOS file did **not** perturb the in-sample features
and the frozen `μ`/`σ` are unchanged.

In [ ]:
REF_PATH = "data/features/f11_macro_context.csv"
try:
    ref = pd.read_csv(REF_PATH, parse_dates=["date"])
    refd = ref.drop_duplicates("date").set_index("date")[cols].sort_index()
    assert set(refd.index).issubset(set(std_panel.index)), "some in-sample dates missing from panel"
    aligned = std_panel.reindex(refd.index)[cols]
    max_abs = float(np.nanmax(np.abs(refd.to_numpy() - aligned.to_numpy())))
    ok = np.allclose(refd.to_numpy(), aligned.to_numpy(), rtol=1e-9, atol=1e-9, equal_nan=True)
    assert ok, f"in-sample values differ (max abs diff {max_abs:.3e})"
    n_oos_only = int((std_panel.index > IN_SAMPLE_END_TS).sum())
    print("PASS — in-sample panel == committed f11_macro_context.csv")
    print(f"  in-sample dates compared : {len(refd)}  |  max abs diff : {max_abs:.3e}")
    print(f"  NEW out-of-sample dates added by the stitch : {n_oos_only}")
except FileNotFoundError:
    print(f"(skipped) {REF_PATH} not present.")

## §6 — Why this is correct, and the leakage check

- **Stitch + freeze-at-2021-07-01** = the OOS macro features are built by the *identical* transform the
  model trained on. The frozen `μ`/`σ` come only from `≤ 2021-07-01` (historical file), so they match
  training; H2-2022 is standardized with those same numbers — no re-fit, no expanding window, no leakage.
- **OOS rows are unfiltered** (no OOS signals yet). To form the model-matrix rows later:
  `panel.set_index("date")` → broadcast onto each instrument's nonzero-signal OOS dates (a `reindex`,
  exactly as `stml.metamodel.pipeline.FeaturePipeline.transform` does).

**Truncation-invariance** — the no-lookahead guarantee: truncating the inputs at a cut leaves every
earlier feature value unchanged.

In [ ]:
cut = pd.Timestamp("2021-01-01")
raw_trunc = {k: v[v.index <= cut] for k, v in raw.items()}
raw45_trunc = mf.assemble_macro_raw(all_dates, raw=raw_trunc)
earlier = raw45.index[raw45.index < cut]
same = np.allclose(raw45.loc[earlier].to_numpy(float),
                   raw45_trunc.loc[earlier].to_numpy(float), equal_nan=True)
print(f"truncation-invariant on the {len(earlier)} dates before {cut.date()}: {same}")
assert same, "truncating future observations changed a past feature value — leakage!"

---
### Outputs

| File | Shape | What it is |
|---|---|---|
| `results/features/f11_macro_panel.csv` | `date` + 45 `f11_*`, 2020→2022-12 | **canonical** cleaned + standardized macro features (full range, incl. OOS) |
| `data/features/f11_macro_context_oos.csv` | `date, instrument` + 45 `f11_*` | OOS slice broadcast to 11 instruments (unfiltered; filter by OOS signals later) |

**Method:** stitch historical + OOS macro → PIT publication lags → as-of panel → `level`+momentum → z-score
**frozen on `≤ 2021-07-01`** (identical to training) → broadcast/filter downstream.